# Fuzzy Inference with `fuzzy_torch`

This notebook is a guided tour of **fuzzy_torch**, a differentiable Fuzzy Inference
System (FIS) library built on PyTorch. By the end you will be able to:

1. describe linguistic variables with **membership functions**,
2. combine them with **t-norms / s-norms** into **fuzzy rules**,
3. build and inspect a **Mamdani** system (the classic tipping problem),
4. build a **Takagi-Sugeno-Kang (TSK)** system of any polynomial order,
5. **train** a fuzzy system with gradient descent, exactly like a neural network,
6. save and load systems as plain **JSON**.

**Why PyTorch?** Every membership-function parameter and every TSK coefficient is an
`nn.Parameter`, and every system is an `nn.Module`. That means a fuzzy system:

- evaluates a whole **batch** of inputs at once,
- moves to the **GPU** with `.to(device)`,
- can be **optimised** with any `torch.optim` optimiser.

> The companion notebook `02_fuzzy_control_envs.ipynb` uses these systems as
> controllers for a crane, a race car and a quadrotor.

## 0. Setup

The cell below works both on **Google Colab** (it installs the library from GitHub)
and **locally** inside a checkout of the repository (it imports the checkout).

In [ ]:
import sys, pathlib, importlib

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip -q install "git+https://github.com/mdrs-thiago/fuzzy_torch.git" matplotlib
else:
    try:
        importlib.import_module('fuzzy_torch')
    except ModuleNotFoundError:
        # The repo is a flat package: its *parent* directory must be on sys.path.
        # This notebook lives in fuzzy_torch/examples/notebooks/.
        sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[2]))

import torch
import matplotlib.pyplot as plt
import fuzzy_torch as ft

torch.manual_seed(0)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
print('fuzzy_torch', ft.__version__, '| torch', torch.__version__)

Everything we need is exported from the top-level package:

In [ ]:
from fuzzy_torch import (
    # membership functions
    TriangularMF, TrapezoidalMF, GaussianMF, BellMF, SigmoidMF,
    # building blocks
    FuzzyVariable, FuzzyRule, TSKConsequent, polynomial_exponents,
    # norms
    Min, Max, Product, ProbabilisticSum, Lukasiewicz,
    # defuzzifiers
    Centroid, Bisector, MeanOfMaximum,
    # systems
    FuzzyModule, MamdaniSystem, TSKSystem,
    # JSON
    build_system, system_to_config, ConfigError,
)

## 1. Membership functions

A membership function (MF) $\mu(x) \in [0, 1]$ says *how much* a crisp value $x$
belongs to a fuzzy set such as "warm" or "fast". `fuzzy_torch` ships five shapes:

| class | parameters | formula |
|---|---|---|
| `TriangularMF(a, b, c)` | feet `a`, `c`, peak `b` | $\max(\min(\frac{x-a}{b-a}, \frac{c-x}{c-b}), 0)$ |
| `TrapezoidalMF(a, b, c, d)` | feet `a`, `d`, plateau `b..c` | $\max(\min(\frac{x-a}{b-a}, 1, \frac{d-x}{d-c}), 0)$ |
| `GaussianMF(mu, sigma)` | centre, width | $e^{-\frac{1}{2}(\frac{x-\mu}{\sigma})^2}$ |
| `BellMF(a, b, c)` | width, slope, centre | $\frac{1}{1 + \lvert\frac{x-c}{a}\rvert^{2b}}$ |
| `SigmoidMF(b, c)` | slope, centre | $\frac{1}{1 + e^{-b(x-c)}}$ |

Every membership function has a `.plot()` method. With no arguments it picks a range
that shows the whole shape; pass `(lo, hi)` to choose one, and `ax=` to draw into an
existing matplotlib axes.

In [ ]:
TriangularMF(2, 5, 8).plot();

In [ ]:
mfs = {
    'TriangularMF(2, 5, 8)':       TriangularMF(2, 5, 8),
    'TrapezoidalMF(1, 3, 6, 9)':   TrapezoidalMF(1, 3, 6, 9),
    'GaussianMF(5, 1.5)':          GaussianMF(5, 1.5),
    'BellMF(2, 3, 5)':             BellMF(2, 3, 5),
    'SigmoidMF(2, 5)':             SigmoidMF(2, 5),
}

fig, axes = plt.subplots(1, 5, figsize=(16, 2.8), sharey=True)
for ax, (name, mf) in zip(axes, mfs.items()):
    mf.plot((0, 10), ax=ax, lw=2)
    ax.set_title(name, fontsize=9)
plt.tight_layout()

### MFs are PyTorch modules

Each shape is an `nn.Module`, and its parameters are `nn.Parameter`s - they show up in
`.parameters()` and will receive gradients.

In [ ]:
tri = TriangularMF(2, 5, 8)
print(tri)
for name, p in tri.named_parameters():
    print(f'  {name} = {p.item():.1f}  (requires_grad={p.requires_grad})')

# MFs work on tensors of any shape - here a (2, 3) batch:
print(tri(torch.tensor([[2.0, 3.5, 5.0], [6.5, 8.0, 9.0]])))

### Shoulder sets

At the ends of a universe we usually want "shoulder" sets that stay at 1. Put two
parameters on top of each other: `TriangularMF(0, 0, 5)` is a left shoulder,
`TrapezoidalMF(7, 9, 10, 10)` a right shoulder. The library handles the degenerate
`a == b` case correctly - the membership at the flat endpoint is exactly 1.

In [ ]:
left = TriangularMF(0, 0, 5)
right = TrapezoidalMF(7, 9, 10, 10)
print('left(0)   =', left(torch.tensor(0.0)).item())
print('right(10) =', right(torch.tensor(10.0)).item())

## 2. Linguistic variables

A `FuzzyVariable` has a **name**, a **range** (universe of discourse) and a set of
named **terms**, each with its own MF. We will use the classic *tipping problem*:

> *How much should I tip, given the quality of the service and of the food?*

In [ ]:
service = FuzzyVariable('Service', 0, 10)
service.add_term('Poor',      TriangularMF(0, 0, 5))
service.add_term('Good',      TriangularMF(0, 5, 10))
service.add_term('Excellent', TriangularMF(5, 10, 10))

food = FuzzyVariable('Food', 0, 10)
food.add_term('Rancid',    TrapezoidalMF(0, 0, 1, 3))
food.add_term('Delicious', TrapezoidalMF(7, 9, 10, 10))

tip = FuzzyVariable('Tip', 0, 30)
tip.add_term('Low',    TriangularMF(0, 0, 15))
tip.add_term('Medium', TriangularMF(0, 15, 30))
tip.add_term('High',   TriangularMF(15, 30, 30))

print(service)
print(food)
print(tip)

`FuzzyVariable.plot()` draws all of its terms:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3))
for ax, var in zip(axes, (service, food, tip)):
    var.plot(ax=ax)
plt.tight_layout()

**Fuzzification** turns a crisp value into a membership degree for every term:

In [ ]:
degrees = service.fuzzify(torch.tensor(6.5))
for term, mu in degrees.items():
    print(f'Service=6.5 is {term:9s} to degree {mu.item():.2f}')

Give `.plot()` a value to see the same thing graphically: the degrees appear as dots
and in the legend.

In [ ]:
service.plot(value=6.5);

## 3. T-norms (AND) and S-norms (OR)

To combine propositions such as *"Service is Poor **AND** Food is Rancid"* we need a
**t-norm**; for OR (and for aggregating rule outputs) we need an **s-norm**.

| t-norm (AND) | formula | s-norm (OR) | formula |
|---|---|---|---|
| `Min` | $\min(a, b)$ | `Max` | $\max(a, b)$ |
| `Product` | $a \cdot b$ | `ProbabilisticSum` | $a + b - ab$ |
| `Lukasiewicz` | $\max(a + b - 1, 0)$ | | |

Below: two overlapping fuzzy sets on the same universe and their combinations.

In [ ]:
grid = torch.linspace(0, 10, 501)
with torch.no_grad():
    a = GaussianMF(4, 1.5)(grid)
    b = GaussianMF(6, 1.5)(grid)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.2), sharey=True)
for ax in axes:
    ax.plot(grid, a, 'k:', lw=1, label='A')
    ax.plot(grid, b, 'k--', lw=1, label='B')
for norm in (Min(), Product(), Lukasiewicz()):
    axes[0].plot(grid, norm(a, b), lw=2, label=type(norm).__name__)
for norm in (Max(), ProbabilisticSum()):
    axes[1].plot(grid, norm(a, b), lw=2, label=type(norm).__name__)
axes[0].set_title('T-norms: A AND B')
axes[1].set_title('S-norms: A OR B')
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()

## 4. Fuzzy rules

A `FuzzyRule` takes an **antecedent** - one `(variable, term)` pair or a list of them,
joined by AND - and a **consequent**:

- `(output_variable, term)` → a **Mamdani** rule,
- a `TSKConsequent` → a **TSK** rule (section 7).

Calling a rule returns its **firing strength** for a dict of inputs.

In [ ]:
r1 = FuzzyRule([(service, 'Poor'), (food, 'Rancid')], (tip, 'Low'))
r2 = FuzzyRule((service, 'Good'), (tip, 'Medium'))
r3 = FuzzyRule([(service, 'Excellent'), (food, 'Delicious')], (tip, 'High'))
r3_prod = FuzzyRule([(service, 'Excellent'), (food, 'Delicious')], (tip, 'High'),
                    t_norm=Product())

for r in (r1, r2, r3):
    print(r)

# A batch of three customers:
batch = {'Service': torch.tensor([1.0, 6.5, 9.0]),
         'Food':    torch.tensor([2.0, 9.8, 8.0])}
print('\nfiring strengths')
print('  r1          ', r1(batch))
print('  r2          ', r2(batch))
print('  r3 (Min)    ', r3(batch))
print('  r3 (Product)', r3_prod(batch))

Mistakes are caught early, with a helpful message:

In [ ]:
try:
    FuzzyRule((service, 'Bad'), (tip, 'Low'))
except KeyError as err:
    print('KeyError:', err)

## 5. A Mamdani system

`FuzzyModule` assembles inputs, output and rules. With Mamdani rules, inference is:

1. **Fuzzify** each input and compute every rule's **firing strength** $w_i$ (t-norm).
2. **Implication**: clip (`Min`) or scale (`Product`) each rule's output set by $w_i$.
3. **Aggregation**: combine the clipped sets across rules (`Max` by default).
4. **Defuzzification**: turn the aggregated set into one crisp number (`Centroid`).

In [ ]:
rules = [r1, r2, r3]
fis = FuzzyModule(inputs=[service, food], output=tip, rules=rules,
                  defuzz_method=Centroid(0, 30))

fis.print_rules()
out = fis({'Service': torch.tensor([6.5]), 'Food': torch.tensor([9.8])})
print(f'\nTip: {out.item():.2f} %')

### Input shapes

Inputs may be Python numbers, 0-dim scalars, `(batch,)` or `(batch, 1)` tensors.
A scalar gives a scalar; anything else gives a `(batch,)` tensor.

In [ ]:
print(fis({'Service': 6.5, 'Food': 9.8}))
print(fis({'Service': torch.tensor(6.5), 'Food': torch.tensor(9.8)}))
print(fis(batch))

### Explaining a decision

`explain()` prints the whole inference chain for the first item of a batch - which
rules fired, how strongly, and why. This *interpretability* is the main selling point
of fuzzy systems.

In [ ]:
fis.explain({'Service': torch.tensor(6.5), 'Food': torch.tensor(9.8)})

### Seeing the inference

`plot_inference` is the graphical version of `explain()`. Each **row is a rule**:

- in the input columns, the term's set is shaded up to the degree the input reaches
  (blank grey cells mean the rule does not use that input);
- `w` is the rule's firing strength (the AND of its degrees);
- in the output column, the consequent set is cut by `w` (implication).

The **bottom row** shows all the terms of each input, and on the right the
**aggregated** output set with the defuzzified (centroid) value in red.

In [ ]:
fis.plot_inference({'Service': 6.5, 'Food': 9.8});

### The control surface

Because the system is batched, we can evaluate a whole 2-D grid of inputs in **one
call**. The resulting surface shows the nonlinear map the rules define.

In [ ]:
fis.plot_surface();

# Count the grid points where no rule fires:
g = torch.linspace(0, 10, 60)
S_, F_ = torch.meshgrid(g, g, indexing='xy')
Z = fis({'Service': S_.reshape(-1), 'Food': F_.reshape(-1)})
print('NaN cells (no rule fired):', torch.isnan(Z).sum().item(), 'of', Z.numel())

Notice the blank corner? With **Service ≈ Poor** and **Food = medium**, *no rule
fires*. There is no meaningful answer, so the system returns `NaN` instead of a
plausible-looking but invented number. You can choose a fallback with `no_rule_value`,
or - better - fix the rule base so it covers the whole input space.

In [ ]:
print('default        :', fis({'Service': 0.0, 'Food': 5.0}))

fis_fallback = FuzzyModule([service, food], tip, rules, no_rule_value=15.0)
print('no_rule_value  :', fis_fallback({'Service': 0.0, 'Food': 5.0}))

# The "proper" fix: a rule base that covers every combination.
food_mid = FuzzyVariable('Food', 0, 10)
food_mid.add_term('Rancid',    TrapezoidalMF(0, 0, 1, 3))
food_mid.add_term('OK',        TriangularMF(1, 5, 9))
food_mid.add_term('Delicious', TrapezoidalMF(7, 9, 10, 10))
rules_full = [
    FuzzyRule([(service, 'Poor'), (food_mid, 'Rancid')],       (tip, 'Low')),
    FuzzyRule([(service, 'Poor'), (food_mid, 'OK')],           (tip, 'Low')),
    FuzzyRule((service, 'Good'),                               (tip, 'Medium')),
    FuzzyRule((food_mid, 'OK'),                                (tip, 'Medium')),
    FuzzyRule([(service, 'Excellent'), (food_mid, 'Delicious')], (tip, 'High')),
    FuzzyRule((food_mid, 'Rancid'),                            (tip, 'Low')),
    FuzzyRule((food_mid, 'Delicious'),                         (tip, 'High')),
]
fis_full = MamdaniSystem([service, food_mid], tip, rules_full)
print('full rule base :', fis_full({'Service': 0.0, 'Food': 5.0}))

fis_full.plot_surface()
Z = fis_full({'Service': S_.reshape(-1), 'Food': F_.reshape(-1)})
print('NaN cells:', torch.isnan(Z).sum().item())

With a single input name, `plot_surface` draws a **slice**: the output along one input
while the others are held fixed.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
for ax, food_value in zip(axes, (2.0, 9.0)):
    fis_full.plot_surface('Service', fixed={'Food': food_value}, ax=ax)
plt.tight_layout()

## 6. Configuring the Mamdani pipeline

```python
FuzzyModule(inputs, output, rules,
            defuzz_method=None,  # Centroid (default), Bisector, MeanOfMaximum
            implication=None,    # Min (default, clipping) or Product (scaling)
            aggregation=None,    # Max (default) or ProbabilisticSum
            tsk_normalize=True,  # TSK: weighted average (True) or weighted sum
            no_rule_value=nan,
            chunk_rules=None)    # performance knob for large rule bases on GPU
```

Let's see how the choices change the control surface. Only `Centroid` is
differentiable: `Bisector` and `MeanOfMaximum` use hard index selection, so the library
warns you when you pick one.

In [ ]:
import warnings

variants = {
    'Min / Max / Centroid':           {},
    'Product / Max / Centroid':       dict(implication=Product()),
    'Min / ProbSum / Centroid':       dict(aggregation=ProbabilisticSum()),
    'Min / Max / Bisector':           dict(defuzz_method=Bisector(0, 30)),
    'Min / Max / MeanOfMaximum':      dict(defuzz_method=MeanOfMaximum(0, 30)),
}
fig, axes = plt.subplots(1, 5, figsize=(22, 3.6))
for ax, (title, kw) in zip(axes, variants.items()):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        sys_ = FuzzyModule([service, food_mid], tip, rules_full, **kw)
    for w in caught:
        print(f'{title}: WARNING -', w.message)
    sys_.plot_surface(n=50, ax=ax)
    ax.set_title(title, fontsize=10)
plt.tight_layout()

for d in (Centroid(0, 30), Bisector(0, 30), MeanOfMaximum(0, 30)):
    print(f'{type(d).__name__:14s} differentiable = {d.differentiable}')

## 7. TSK (Takagi-Sugeno-Kang) systems

In a TSK system the consequent of each rule is a **crisp function** of the inputs
instead of a fuzzy set:

$$\text{IF } x_1 \text{ is } A_1 \text{ AND } \dots \text{ THEN } y = f_i(x)$$

and the output is the firing-strength-weighted average

$$y = \frac{\sum_i w_i\, f_i(x)}{\sum_i w_i}.$$

No defuzzification is needed, and the output variable needs only a range.

`TSKConsequent(order=p)` makes $f_i$ a **full multivariate polynomial** of degree
$\le p$ (cross terms included). For $n$ inputs this has $\binom{n+p}{p}$ coefficients.

In [ ]:
import math
for order in range(4):
    exps = polynomial_exponents(2, order)
    names = []
    for e in exps.tolist():
        t = '·'.join(f'{v}' + (f'^{k}' if k > 1 else '') for v, k in zip('ab', e) if k)
        names.append(t or '1')
    print(f'order {order}: {math.comb(2 + order, order):2d} terms ->', ', '.join(names))

By default (`normalize=True`) each input is rescaled from its variable range onto
$[-1, 1]$ before taking powers, which keeps high orders well conditioned. Pass
`normalize=False` to have coefficients in raw units.

Here is a hand-written zero-order TSK version of the tipping problem (each rule
outputs a constant tip):

In [ ]:
tip_tsk = FuzzyVariable('Tip', 0, 30)      # range only - no terms needed
tsk_rules = [
    FuzzyRule([(service, 'Poor'), (food_mid, 'Rancid')],         TSKConsequent(0, coefficients=[3.0])),
    FuzzyRule((service, 'Good'),                                 TSKConsequent(0, coefficients=[15.0])),
    FuzzyRule((food_mid, 'OK'),                                  TSKConsequent(0, coefficients=[13.0])),
    FuzzyRule([(service, 'Excellent'), (food_mid, 'Delicious')], TSKConsequent(0, coefficients=[27.0])),
    FuzzyRule((food_mid, 'Rancid'),                              TSKConsequent(0, coefficients=[5.0])),
    FuzzyRule((food_mid, 'Delicious'),                           TSKConsequent(0, coefficients=[22.0])),
]
tsk = TSKSystem([service, food_mid], tip_tsk, tsk_rules)
print('mode:', tsk.mode)
tsk.print_rules()
tsk.explain({'Service': 6.5, 'Food': 9.8})

tsk.plot_surface();

For a TSK system `plot_inference` marks each rule's crisp output `f_i(x)` as a bar
whose height is its firing strength; the result is their weighted average.

In [ ]:
tsk.plot_inference({'Service': 6.5, 'Food': 9.8});

> `FuzzyModule` infers the mode from the rules. `MamdaniSystem` and `TSKSystem` are the
> same class plus an assertion, for when you want the intent stated in code. A rule
> base may not mix both kinds of consequent.

In [ ]:
try:
    FuzzyModule([service, food_mid], tip, [rules_full[0], tsk_rules[0]])
except ValueError as err:
    print('ValueError:', err)

## 8. Learning from data (ANFIS-style)

Since everything is differentiable, we can **fit** a fuzzy system to data with
gradient descent - both the TSK coefficients *and* the membership functions move.

Target: a noisy nonlinear function $y = \sin(x) + 0.3x$ on $[0, 2\pi]$.

In [ ]:
torch.manual_seed(0)
X_train = torch.rand(300) * 2 * math.pi
y_train = torch.sin(X_train) + 0.3 * X_train + 0.05 * torch.randn(300)
X_test = torch.linspace(0, 2 * math.pi, 400)
y_true = torch.sin(X_test) + 0.3 * X_test

def make_learner(n_terms=5, order=1):
    xv = FuzzyVariable('x', 0, 2 * math.pi)
    centres = torch.linspace(0, 2 * math.pi, n_terms)
    width = (2 * math.pi) / (n_terms - 1) / 2
    for i, c in enumerate(centres):
        xv.add_term(f'A{i}', GaussianMF(c.item(), width))
    yv = FuzzyVariable('y', -2, 3)
    rules = [FuzzyRule((xv, f'A{i}'), TSKConsequent(order=order)) for i in range(n_terms)]
    return TSKSystem([xv], yv, rules), xv

model, xvar = make_learner()
print('trainable parameters:', sum(p.numel() for p in model.parameters()))
for name, p in list(model.named_parameters())[:4]:
    print(f'  {name:55s} {tuple(p.shape)}')
print('  ...')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.4))
xvar.plot(ax=axes[1], ls='--', legend=False)   # MFs before training

opt = torch.optim.Adam(model.parameters(), lr=0.02)
losses = []
for epoch in range(800):
    opt.zero_grad()
    loss = ((model({'x': X_train}) - y_train) ** 2).mean()
    loss.backward()
    opt.step()
    losses.append(loss.item())

print(f'final MSE: {losses[-1]:.4f}')
axes[0].semilogy(losses); axes[0].set_title('training loss (MSE)'); axes[0].set_xlabel('epoch')
xvar.plot(ax=axes[1])                          # MFs after training
axes[1].set_title('membership functions (dashed = before)')
with torch.no_grad():
    axes[2].scatter(X_train, y_train, s=5, color='gray', alpha=0.5, label='data')
    axes[2].plot(X_test, y_true, 'k--', label='true function')
    axes[2].plot(X_test, model({'x': X_test}), 'r', lw=2, label='TSK fit')
axes[2].legend(fontsize=8); axes[2].set_title('fit')
plt.tight_layout()

The learned rules are still readable - each says *"around here, the function looks
like this line"*:

In [ ]:
model.print_rules()
model.plot_inference({'x': 2.0});

### Tuning a Mamdani system

Mamdani systems with `Centroid` defuzzification are differentiable too. Suppose a
restaurant owner has their *own* notion of a low / medium / high tip. We only observe
the tips they give (here produced by a hidden "teacher" system with different `Tip`
sets). Can we recover their notion by tuning the `Tip` membership functions, while
keeping our rules?

In [ ]:
import copy

expert = copy.deepcopy(fis_full)
tuned = copy.deepcopy(fis_full)

# The hidden teacher: same rules, different Tip sets.
teacher = copy.deepcopy(fis_full)
with torch.no_grad():
    for mf, (a_, b_, c_) in zip(teacher.output_variable.terms.values(),
                                [(0, 5, 12), (8, 17, 24), (18, 25, 30)]):
        mf.a.fill_(a_); mf.b.fill_(b_); mf.c.fill_(c_)

torch.manual_seed(1)
S = torch.rand(400) * 10
F = torch.rand(400) * 10
with torch.no_grad():
    target = teacher({'Service': S, 'Food': F})

# Optimise only the output sets: the rules and input sets stay as the expert wrote them.
opt = torch.optim.Adam(tuned.output_variable.parameters(), lr=0.02)
for step in range(400):
    opt.zero_grad()
    loss = ((tuned({'Service': S, 'Food': F}) - target) ** 2).mean()
    loss.backward()
    opt.step()
    if step % 100 == 0 or step == 399:
        print(f'step {step:3d}  MSE {loss.item():.4f}')

fig, axes = plt.subplots(1, 3, figsize=(16, 3))
for ax, sys_, title in zip(axes, (expert, tuned, teacher),
                           ('expert (start)', 'tuned', 'hidden teacher')):
    sys_.output_variable.plot(ax=ax)
    ax.set_title(f'Tip terms - {title}')
plt.tight_layout()

The tuned sets move towards the teacher's, and the error drops by more than an order of
magnitude. They need not match exactly: several different sets can produce nearly the
same centroids (the problem is not fully *identifiable*).

**Try it:** raise the learning rate to `0.1`. Training diverges - a triangle whose
feet cross its peak is no longer a sensible fuzzy set. Constraining parameters (or a
smaller learning rate) is part of designing a trainable fuzzy system.

## 9. Systems as JSON

A system can also be written as plain data. This is how students submit controllers
for the Grand Prix in notebook 02. MFs are `[shape, *params]` with shape one of
`tri`, `trap`, `gauss`, `bell`, `sigmoid`. A rule's `then` picks the mode:

- a term name → Mamdani,
- a number → zero-order TSK,
- a list `[c0, c1, ...]` → first-order TSK,
- `{"order": p, "coefficients": [...]}` → any order.

In [ ]:
spec = {
    "inputs": {
        "error": {"range": [-1, 1], "terms": {
            "neg":  ["trap", -1, -1, -0.5, 0],
            "zero": ["tri", -0.5, 0, 0.5],
            "pos":  ["trap", 0, 0.5, 1, 1]}}
    },
    "output": {"name": "u", "range": [-1, 1], "terms": {
        "down": ["tri", -1, -1, 0],
        "hold": ["tri", -1, 0, 1],
        "up":   ["tri", 0, 1, 1]}},
    "rules": [
        {"if": {"error": "neg"},  "then": "up"},
        {"if": {"error": "zero"}, "then": "hold"},
        {"if": {"error": "pos"},  "then": "down"},
    ],
}
controller = build_system(spec)
controller.print_rules()

e = torch.linspace(-1, 1, 200)
with torch.no_grad():
    plt.figure(figsize=(5, 3))
    plt.plot(e, controller({'error': e}))
    plt.xlabel('error'); plt.ylabel('u'); plt.title('JSON-built controller');

Errors in a spec point at the offending field, so they can be shown to the author as-is:

In [ ]:
bad = copy.deepcopy(spec)
bad['rules'][1]['if']['error'] = 'tiny'
try:
    build_system(bad)
except ConfigError as err:
    print('ConfigError:', err)

And the reverse: `system_to_config` exports any system - including a **trained** one -
back to JSON, ready to save or share.

In [ ]:
import json
cfg = system_to_config(model, digits=3)
for key, value in cfg.items():
    if key == 'rules':
        print('rules:')
        for r in value:
            print('   ', json.dumps(r))
    else:
        print(f'{key}:', json.dumps(value))

# Round trip: rebuild the trained model from its JSON and check it computes the same.
rebuilt = build_system(system_to_config(model))
with torch.no_grad():
    diff = (rebuilt({'x': X_test}) - model({'x': X_test})).abs().max()
print('\nmax difference after round trip:', diff.item())

## 10. Running on a GPU

A `FuzzyModule` is a normal `nn.Module`, so `.to(device)` moves every parameter.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu_model = copy.deepcopy(model).to(device)
xs = torch.linspace(0, 2 * math.pi, 100_000, device=device)
with torch.no_grad():
    ys = gpu_model({'x': xs})
print(f'evaluated {xs.numel():,} inputs on {ys.device}')

## 11. Exercises

1. **Coverage.** Add a `Fair` term to `Service` and write rules for it. Re-plot the
   control surface - are there still `NaN` regions?
2. **Shapes.** Replace every triangular MF in the tipping problem with a `GaussianMF`.
   How does the surface change? Why is it smoother?
3. **Operators.** Compare `Min` vs `Product` as the AND t-norm (`FuzzyRule(...,
   t_norm=Product())`) on the surface.
4. **TSK order.** Re-run section 8 with `order=0` and `order=2`, and with 3 vs 9 terms.
   Plot test error vs number of parameters. When does it overfit?
5. **Interpretability.** After training, call `model.explain({'x': 2.0})`. Do the
   learned rules still make sense? What happened to the MF widths?
6. **JSON.** Write the complete tipping system (`fis_full`) as a JSON spec by hand
   and check that `build_system` gives the same output as the Python version.